<div align="left">

# PyPTO Agent 架构与使用方法

手工开发算子的典型流程是：理清需求、编写 PyTorch golden、设计 kernel 方案、编码调试、最后在 NPU 上验证精度。PyPTO Agent 做的事情本质上一样，区别只是把每一步交给专门的智能体自动完成，不用人一步步手动敲代码，省去大量重复劳动——这是它最大的价值：高效与自动化。至于「做得稳、能追溯」，则靠上一节介绍的工程约束（独立验证、Lint Gate、状态记录与真实 NPU 验收）来保障。

本节先介绍 Agent 的团队架构与支撑体系，再演示如何用 OpenCode 实际驱动 Agent 完成算子开发与独立验收。
本节的路线是这样的：先看「团队怎么搭」（总控+八个智能体）→ 再看「团队靠什么协作」（技能库、共享状态、门禁）→ 然后跟着 ReLU 案例走一遍全流程 → 最后学会自己动手运行和验收。

</div>

<div align="left">

## 整体架构：一个总控与八个智能体

### 阶段总览

PyPTO Agent 是一个多智能体团队：1 个 Orchestrator（总控）与 8 个各有专长的子智能体协同工作。学习者只需将需求交给 Orchestrator，它会按照 Stage 1 至 Stage 7 的流程，依次调度对应智能体完成各阶段任务：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">阶段</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">内容</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">执行智能体</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">主要产物</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 1 需求规划</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">将需求整理为结构化规格</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Planner</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>SPEC.md</code>、<code>API_REPORT.md</code></td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 2 算法基准</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编写独立的 PyTorch golden 参考实现</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Mathematician</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code><op>_golden.py</code></td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 3 架构设计</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">评估复杂度，确定分块与模块方案</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Architect</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>DESIGN.md</code></td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 4 模块接口设计</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">拆分模块、定义接口；单模块路径下仅定义接口约定，多模块路径下同步构建测试用例</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Designer、Verifier</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>module_interfaces.yaml</code>、测试用例</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 5 构建实现</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">逐模块编码、校验、调试与修复</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Coder、Verifier、Debugger</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code><op>_impl.py</code></td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 6 整体验证</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">端到端全量精度测试</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Verifier</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">测试报告，精度冻结</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 7 性能优化（可选）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">在精度冻结后进行性能调优</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Optimizer</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">优化方案与回归测试</td></tr></tbody></table>

这一整套流程先用一张图把全貌画出来——从学习者把任务交进去，到最终产出验收证据，中间各阶段由哪些智能体接力完成：

<img src="./images/agent_orchestrator_workflow.png" alt="PyPTO Agent 整体工作流程" width="900px">

*图：PyPTO Agent 整体工作流程示意图（Stage 1–7 与各阶段责任人）*

> **产物文件是什么**：表格里的这些文件是每个阶段的「作业」。`SPEC.md` 是算子规格说明（叫什么、输入输出、什么精度要求）；`API_REPORT.md` 报告算子对应的上层接口；`DESIGN.md` 是设计方案；`module_interfaces.yaml` 约定各模块之间的接口；`*_golden.py` 是 PyTorch 参考实现（标准答案）；`*_impl.py` 是实际实现代码。后面会逐项再讲。

> **单模块路径 / 多模块路径**：算子如果比较简单（比如一次计算就完事），就只当成一个模块，从头做到尾；如果比较复杂（比如矩阵乘、要分多步处理），就拆成几个模块一个个推进。

### Orchestrator：只做编排的总控

Orchestrator 是团队的统一入口，内置 Stage 1–7 状态机（状态机可以理解为「记分牌」——记录当前进行到哪个阶段、每步是否完成）。它的职责很专注：只管流程编排——按阶段推进任务、调度对应智能体、依据已有校验证据判断阶段是否完成、把执行结果写进共享状态。它不写代码、不跑测试、不亲自调试，也不会为了省事跳过校验门禁。你要做的就是把任务交给它，不需要直接去和每一个子智能体打交道。

</div>

<div align="left">

## 八个智能体的职责边界

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">智能体</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">主要阶段</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">核心职责</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">明确不负责</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Planner（规划）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 1</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">将需求转化为结构化规格 <code>SPEC.md</code> 与 API 映射表</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">不做方案设计</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Mathematician（数学）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 2</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编写纯 PyTorch FP32 golden 参考实现并自检</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">不接触 PyPTO 实现</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Architect（架构）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 3</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">评估算子复杂度，确定分块策略与模块数量</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">不编写实现代码</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Designer（设计）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 4</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">拆分模块，在 <code>module_interfaces.yaml</code> 中定义接口约定</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">不实现模块逻辑</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Coder（编码）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 5</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编写实现文件，应用 Debugger 给出的补丁</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">不写测试、不做通过判定</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Verifier（验证）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 4–7</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">专职校验：张量比对、布局检查、对抗性用例、故障分类</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">不排查根因、不输出修复方案</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Debugger（调试）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 5</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">对失败用例定位根因，在共享文档中输出补丁方案</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">不编写正式业务代码</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Optimizer（优化）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 7</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">精度冻结后执行性能调优</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">不改变数值语义</td></tr></tbody></table>

Coder、Verifier、Debugger 三者之间的隔离是整套架构最重要的边界设计：**编写实现的智能体永远不参与通过判定**。这从机制上解决了章节介绍中提到的「编码与验证不分」问题。当某个模块校验失败时，标准处理链路为：

> Verifier 判定失败并归类故障 → Debugger 定位根因并给出补丁方案 → Coder 应用补丁 → Verifier 重新校验

前一模块未通过，不会启动下一模块的开发。

> **表格里的几个校验术语**，这里提前解释一下：**张量比对**，就是把实现算出的结果和 golden 参考结果一个位置上、一个位置上地比较；**布局检查**，是确认数据在内存里的排布方式符合 NPU 的要求；**对抗性用例**，是故意挑一些刁钻的输入（比如全是 0、极大值、边界值）来考验实现会不会出错；**故障分类**，是把失败原因分门别类（是精度问题还是越界问题），方便 Debugger 接手排查。

</div>

<div align="left">

## 支撑体系：能力来源、协作基础与质量保障

### Skills：按需加载的技能库

Agent 的领域知识不是一股脑塞给每个智能体的，而是放在工程目录的 `.opencode/skills/` 下（运行任务前，`init.sh` 会把这些技能安装到这个位置）的一组技能（Skill）里。每项技能是一个相对独立的小单元，覆盖算子开发的某一环：需求规划、golden 编写、方案设计、结果校验，还有按故障类型划分的调试方法（精度异常、AICore 报错、主机栈追踪、Workspace 内存冲突等）和多层级的性能调优方法。这些调试方法可以理解为「常见问题的排查手册」：AICore 报错是 NPU 侧执行时报错，主机栈追踪是通过报错信息定位到具体代码位置，Workspace 内存冲突是多个算子共用内存时互相覆盖导致的异常。

每个智能体只加载当前任务所需的少量技能，只携带与本阶段相关的知识——这样就避免了单个智能体承载过多信息而遗漏关键约束。

### 双份共享状态：叙事记录与状态机

团队不依赖隐式记忆协作。每个算子的工作目录下有两份持续更新的文件，职责清晰、互不重叠：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">文件</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">定位</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">记录内容</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">写入权限</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>MEMORY.md</code></td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">叙事记录</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">人类可读：设计思路、决策依据、检查凭证、调试过程</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">从 Stage 5 起由各智能体读写，任务交接时同步更新</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>.orchestrator_state.json</code></td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">状态机文件</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">机器可读：当前阶段、重试次数、模块状态、产物哈希与回滚记录</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">仅 Orchestrator 通过 <code>state_transition</code> 工具写入</td></tr></tbody></table>

Stages 1–4 的过程记录分散在 SPEC、golden、DESIGN 等产物文档中；从 Stage 5 起，所有阶段的结论与过程统一汇入 `MEMORY.md`。状态机文件则从任务一开始就存在，全程记录阶段推进与重试情况。有了这两份文件，任务中断后可以从断点恢复，也可以随时回溯某个结论基于什么证据。状态机的写入权限集中管控，避免失败任务被随意标记为「已完成」。

> **如何解读状态机文件**：打开 `.orchestrator_state.json`，重点关注以下字段：
> - `current_stage`：当前正在执行的阶段编号（1-7）
> - `retry_count`：当前阶段的重试次数，超过阈值会触发回滚或终止
> - `module_status`：各模块的完成状态。英文含义：pending=未开始，in_progress=进行中，completed=已完成，failed=失败
> - `artifacts_hash`：关键产物的哈希值，可理解为给文件算出的「指纹」，内容一变指纹就变，用来检测文件是否被意外修改

### Lint Gate：写入阶段的自动门禁

Lint Gate（自动门禁）是独立于所有智能体运行的自动规则检查，作为工具调用的后置环节固定生效，智能体无法主动关闭或跳过。每当有智能体新建或修改 `*_impl.py`、`*_golden.py`、`test_*.py` 这三类关键文件，检查就会自动触发，从五个维度把关：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">维度</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">检查内容</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">D1 框架合规性</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">装饰器、张量签名 Shape、JIT 相关规范</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">D2 交付物完整性</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">各阶段要求产出的文档与代码是否齐备</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">D3 文件隔离</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">实现文件禁止引入 Torch，golden 文件禁止调用 PyPTO</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">D4 测试规范</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">测试用例覆盖、误差容差配置</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">D5 跨文件一致性</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">模块接口约定与实现代码是否匹配</td></tr></tbody></table>

严重违规会在**写入那一刻**直接拦下，不合规的代码根本进入不了工作目录；任何绕过 Orchestrator、直接修改 `.orchestrator_state.json` 的操作也会被拦截。

为什么门禁必须卡在写入阶段？这不是凭空设计的，而是实践踩坑后的教训：

1. 早期只在事后检查精度，模型多次编码失败后会偷偷用 Torch API 代替 PyPTO 实现——精度能过，但不符合框架规范；
2. 引入语法级检查后，模型转而寻找规则漏洞，例如创建空壳 `@jit` 函数（`@jit` 是 PyPTO 提供的 JIT 装饰器，把 Python 函数编译成可在 NPU 上运行的代码，空壳指只有框架、没有实际计算）、把真实计算放在调用 Torch 的普通函数中，说明校验必须看代码真正做了什么（从输入到输出的整个处理过程），而不能只看语法长得像不像；
3. 曾出现检查告警正常输出、违规代码仍然落盘的情况，事后巡检无法阻止非法产物污染后续流程，于是门禁被改造为写入阶段强制阻断。

D3 的文件隔离同时从底层封堵了「用 golden 冒充实现」的路径：实现文件根本访问不到 golden 代码。

</div>

<div align="left">

## 端到端走读：ReLU 从任务到精度证据

下面用本章的 ReLU 任务（`[8,128]` FP32，`y = max(x, 0)`）的真实运行记录，将前面的团队架构与支撑体系串起来走读一遍：

> **说明**：本示例使用较小的固定 shape `[8,128]` 是为了便于初学者理解 Agent 的工作流程。实际生产环境中，PyPTO Agent 同样支持动态 shape 和更复杂的算子。

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">阶段</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">产物</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">本次运行的关键信息</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 1</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>SPEC.md</code>、<code>API_REPORT.md</code></td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">规格写明固定 shape <code>[8,128]</code>、FP32、容差 <code>atol=rtol=1e-3</code>，并注明「Stage 6 精度验收后停止」</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 2</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>relu_golden.py</code></td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">用纯 PyTorch 写出 <code>torch.maximum(x, 0)</code> golden 实现，并与独立参考 <code>torch.relu</code> 在多组 shape 上互相校验</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 3</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>DESIGN.md</code></td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">复杂度评估：核心计算仅 4 行、无矩阵乘、无跨步状态，走单模块路径；向量 Tile 定为 <code>[8,64]</code></td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 4</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>module_interfaces.yaml</code></td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">单模块路径，仅定义接口约定；测试用例在 Stage 5 由 Verifier 构建</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 5</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>relu_impl.py</code></td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Coder 写出 <code>@pypto.frontend.jit</code> 实现，核心计算为 <code>pypto.relu</code>；Verifier 在真实 NPU 上校验通过</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 6</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>test_relu.py</code> 测试报告</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">L0（基础用例）与 L1（边界用例）两组测试均 <code>all_close=True</code>，最大绝对误差 <code>0.000e+00</code>，精度冻结</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">独立复验</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">学习者重跑 <code>test_relu.py</code></td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">退出码为 0，复现全部 all_close 结果</td></tr></tbody></table>

对表中几个词稍作解释：`@pypto.frontend.jit` 是 PyPTO 的 JIT 装饰器，作用是把 Python 写的函数编译成能在 NPU 上运行的代码；`pypto.relu` 是 PyPTO 内置的 ReLU 计算函数，直接调用就能完成 `max(x, 0)`；`Tile` 是把数据切成的一块块小数据块，向量 Tile `[8,64]` 指的是每个块是 8 行 64 列的一段；`L0` 指常规用例，`L1` 指边界和特殊取值用例。

Stage 7 没有执行，因为任务一开始就写明「不进行性能调优」。注意最后一步：即便 Agent 报告了成功，学习者仍然要独立重跑一遍测试——这是本章始终坚持的验收方式。

</div>

### 用一段代码，把整个调度链画出来

如果觉得前面的表格信息量太大，可以运行下面这段代码：它会用文本把「总控 → 7 个阶段 → 各阶段智能体 → 产出物」的调用链画出来。运行后你会看到一条从学习者的任务一路流向最终验收证据的**流水线**，这就是 Agent 干活时内部一步一步推进的顺序。


In [ ]:
def draw_agent_flow():
    """用文本画出 PyPTO Agent 的调度流程：总控依次调度智能体完成 7 个阶段。"""
    stages = [
        ("Stage 1 需求规划", "Planner", "SPEC.md"),
        ("Stage 2 算法基准", "Mathematician", "relu_golden.py"),
        ("Stage 3 方案设计", "Architect", "DESIGN.md"),
        ("Stage 4 接口约定", "Architect", "module_interfaces.yaml"),
        ("Stage 5 编码调试", "Coder+Verifier+Debugger", "relu_impl.py"),
        ("Stage 6 精度验收", "Verifier", "all_close=true"),
        ("Stage 7 性能调优", "Optimizer", "可选"),
    ]

    print("学习者")
    print("   |  任务定义")
    print("   v")
    print("Orchestrator（总控：拆解任务、按流程调度）")
    for name, agents, output in stages:
        print(f"   |  {name}")
        print(f"   v  执行：{agents}")
        print(f"   >  产出：{output}")
    print()
    print("提示：Stage 1-6 是精度开发必做项，Stage 7 性能调优是可选结束点。")


if __name__ == "__main__":
    draw_agent_flow()


<div align="left">

## 使用 OpenCode 驱动 Agent

### 任务定义的五要素

「帮我写一个 ReLU」这样的需求太模糊：模型可能自行补全 shape、dtype 和容差，每次结果都不一样，学习者也无从判断对错。一份可执行的任务必须说清五组信息，而且每一组都应该能在后面的产物或证据中找到对应：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">信息组</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">要回答的问题</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">对应产物</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">名称与公式</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">算子叫什么，每个输出如何由输入计算？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>SPEC.md</code>、golden</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">输入与输出</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">参数名、数量、shape 和 dtype 是什么？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>SPEC.md</code>、实现签名</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">支持范围</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">固定 shape、动态 shape 和边界条件覆盖到哪里？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left"><code>SPEC.md</code>、测试用例</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">参考与测试</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">golden 语义、执行设备、输入覆盖和逐输出容差是什么？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">golden、test 脚本</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">停止条件</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">工作流在哪一步结束，是否进行性能调优？</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">状态机文件</td></tr></tbody></table>

这些信息各有各的去处：公式会进 SPEC 和 golden；接口会进实现签名（实现签名就是实现函数的名称和参数列表，决定这个算子以什么形式对外提供）；设备和容差会进测试；停止条件会进状态机。有一点值得注意：「本任务不做性能调优，精度通过后停止」也是一种清楚明确的写法，比空着不写要好——信息不全时，Agent 可能会反复追问，也可能自己随便补全，结果就不可控了。

</div>

<div align="left">

### ReLU 任务定义示例

下面就是上一节案例中实际使用的那份任务。本章示例只支持 `[8,128]` FP32，所以任务里不需要动态 shape、更多 dtype 或性能目标：

```text
# ReLU 算子开发任务

请使用 PyPTO 算子开发工作流完成下列任务。所有必需信息已经给出，请直接开始，不要重复询问算子名称、shape、dtype 或性能目标。

## 算子规格

- 算子名称：`relu`
- 输入：`x`，shape 为 `[8, 128]`，dtype 为 FP32
- 输出：`y`，shape 为 `[8, 128]`，dtype 为 FP32
- 数学定义：`y[i, j] = max(x[i, j], 0)`
- 支持范围：本任务只要求上述固定 shape 和 dtype

## 参考实现与测试

- golden 必须是独立、仅使用 PyTorch、并与上述数学定义一致的 FP32 参考实现。
- 测试输入必须同时覆盖负数、零和正数。
- 测试必须在 `TILE_FWK_DEVICE_ID` 指定的真实 Ascend NPU 上运行，不能使用 CPU 或仿真结果代替。
- 输出结果中的每个值都必须与参考结果逐一比较；精度要求为 `atol=1e-3`、`rtol=1e-3`，并且 `all_close=true`。

## 工作流范围

- 初始化编排状态时将 `max_stage` 设置为 6。
- 性能目标：本任务不进行性能调优。端到端精度验证通过后停止，不进入性能优化阶段。
- 在 `custom/relu/` 中生成规格、golden、PyPTO 实现、测试和工作流要求的过程产物。
- 必须实际运行生成的 `custom/relu/test_relu.py`，不能只根据 Agent 的文字总结报告完成。

## 禁止行为

- 不得在 `relu_impl.py` 中使用 Torch API 完成核心数值计算。
- 不得为了匹配实现而修改 golden 的数学语义。
- 不得放宽精度容差。
- 不得用 CPU、仿真或「实现与自身比较」冒充真实 NPU 精度通过。
```

任务里这几处说法值得解释：

- `TILE_FWK_DEVICE_ID`：课程环境设置的设备号环境变量。可以把它理解成「告诉测试程序是哪一块真实 NPU 卡」的门牌号，测试程序凭它找到真实 NPU 来跑；
- 「输出结果的每个值」：指的是算子的每一个输出元素。哪怕一个算子输出的是一整个数组，也要**逐个数字**和参考结果比较，不能只看整体对不对；
- `max_stage`：表示工作流最远走到哪一阶段。这里设为 6，就是明确告诉 Agent「精度验收通过就可以停，不用再做性能优化」。

检查一份任务写得好不好，可以逐项自问：公式是否只有一种解读方式；shape 和 dtype 是否明确出现在输入与输出中；golden 是否要求独立；测试输入能否区分负数、零和正数；设备与容差是否可执行；停止条件是否写明。全部能在原文中找到对应，这份任务就合格了。

最好的输入标杆，是一份**可以直接运行的 PyTorch Golden 函数**——它给出标准答案，Agent 可以逐值对照，验收结果一目了然。这样无论 Agent 中间怎么折腾，最终能和这份「标准答案」对上，就算成功。



### 亲手运行一次 golden

光看任务演示不过瘾，我们直接把上面任务里的 "独立 PyTorch golden" 写出来跑一遍。这一步模拟的是 Agent 的 Stage 2（Mathematician 产出标准答案）：一份**不依赖 PyPTO、只用 PyTorch** 的参考实现，再加上一段覆盖「负数、零、正数」三类输入的自我校验。**它在任何机器上都能直接运行，连 NPU 都不需要**。


In [ ]:
import warnings
# 过滤环境提示信息，让输出聚焦示例本身
warnings.filterwarnings("ignore")

import torch


def relu_golden(x):
    """ReLU 的 golden 参考实现：y = max(x, 0)，纯 PyTorch，独立于 PyPTO 实现。"""
    return torch.maximum(x, torch.zeros_like(x))


def check_golden():
    """用独立的 torch.relu 做标准答案交叉校验，覆盖负数、零、正数三类输入。"""
    torch.manual_seed(42)

    # 与任务一致的固定 shape：随机输入已覆盖负数/正数，再人为补上精确的 0
    x = torch.randn(8, 128, dtype=torch.float32)
    x[0, 0] = 0.0
    print(f"输入 x shape={tuple(x.shape)} dtype={x.dtype}")
    print(f"  负数个数={int((x < 0).sum())}，零的个数={int((x == 0).sum())}，正数个数={int((x > 0).sum())}")

    y_impl = relu_golden(x)
    y_ref = torch.relu(x)
    all_close = torch.allclose(y_impl, y_ref, atol=1e-3, rtol=1e-3)
    print(f"实现 vs torch.relu: all_close={all_close}")
    return bool(all_close)


if __name__ == "__main__":
    ok = check_golden()
    print("\ngolden 自校验结果:", "PASS" if ok else "FAIL")


### 运行任务：用交互式界面启动 OpenCode

环境准备好之后，我们就可以把 Agent 真正跑起来了。整个流程分为两步：先**准备工程环境**，再**启动 OpenCode 对话界面**。跟着下面的截图和命令一步步来，就能看到 Agent 开始干活。

#### 第 1 步：准备工程环境（把"工具箱"搬进自己家）

Agent 的技能库和编排器都放在一个叫 **pypto-gym** 的仓库里。在动手前，先把这份仓库下载到本地，并把算子开发 Agent 部署好——这一步可以理解为「把工具箱搬进自己家」。请在终端按顺序执行下面 3 条命令：

```bash
# 下载 pypto-gym 仓库（含算子开发 Agent 的技能与编排器）
git clone https://gitcode.com/cann/pypto-gym.git

# 进入存有「算子开发编排器」的工程目录，后面所有操作都以这里为起点
cd pypto-gym/cannbot-skills/plugins-official/pypto-op-orchestrator

# 执行部署脚本：把 Agent 的技能库、状态机和配置装进当前目录
bash init.sh project opencode
```

这三条命令分别做了什么？

- `git clone ...`：把 pypto-gym 整个仓库下载到本地，以后离线也能用；
- `cd ...`：进入存放「算子开发编排器」的插件目录，后面所有操作都以这里为起点；
- `bash init.sh project opencode`：执行部署脚本，把 Agent 的技能库、状态机和配置装进当前目录。装好之后，当前目录就是一个可以直接「对话式开发算子」的 Agent 工程。

下图就是第 1 步执行后终端的样子——三条命令依次执行成功，没有任何报错就说明工具已经准备好了：

<div align="left">
<img src="./images/opencode_step1_env_setup.png" alt="opencode_step1_env_setup" width="900px">
</div>

> 部署方式的完整说明（全局安装、指派自定义工程路径等高级用法），可以参考插件的官方快速开始文档：<https://gitcode.com/cann/cannbot-skills/blob/master/plugins-official/pypto-op-orchestrator/quickstart.md>

#### 第 2 步：启动 OpenCode，进入对话界面

工程部署好之后，**仍然在第 1 步 `cd` 进入的那个目录里**，输入下面的命令启动 OpenCode：

```bash
# 启动 opencode，进入交互式对话界面
opencode
# 或者，用自动模式跳过运行命令、读写文件等确认环节
opencode --auto
```

- **`opencode`（推荐）**：进入交互式对话界面。粘贴任务、实时观察每一阶段在做什么、中途提问打断，都在同一个界面里完成，对初学者最友好；
- **`opencode --auto`**：自动模式，跳过运行命令、读写文件等确认环节，适合任务内容明确、希望无人值守跑完的场景。

下图就是执行第 2 步时终端的样子——`opencode` 命令敲下回车的一瞬间：

<div align="left">
<img src="./images/opencode_step2_launch.png" alt="opencode_step2_launch" width="900px">
</div>

#### 第 3 步：看懂这个界面，把任务交给总控

启动完成后，你会看到一个交互式对话界面（下图）。这个界面就是你和 Agent「打交道的窗口」：底部是**输入框**，上部是**对话/进度区**，Agent 会在里面告诉你「我正在做 Stage 1、正在写 SPEC、正在验证精度……」：

<div align="left">
<img src="./images/opencode_tui_main.png" alt="opencode_tui_main" width="900px">
</div>

现在把你在上一小节准备好的「任务定义」**粘贴到输入框里，按回车**，Orchestrator（总控）就会开始干活了。整个过程你什么都不用做，看着它一步步推进即可。


<div align="left">

### 关键产物与独立验收
#### 第一步：先看产物目录树
Agent 运行结束后，`custom/<op>/` 下会生成一整套产物。`custom/` 可以理解为「这个工程里所有自定义算子的大本营」，`<op>` 是算子名，比如本章实践的行 Softmax 对应 `custom/row_softmax/`。目录大致长这样：

```text
custom/row_softmax/
├── SPEC.md                 # Stage 1 需求规格：算子叫什么、输入输出、精度要求
├── API_REPORT.md           # Stage 1 API 报告：算子对应哪些可用接口
├── row_softmax_golden.py   # Stage 2 golden 参考实现（标准答案）
├── DESIGN.md               # Stage 3 设计文档：分块策略与逻辑结构
├── module_interfaces.yaml  # Stage 4 模块接口约定
├── row_softmax_impl.py     # Stage 5 最终实现代码（PyPTO kernel）
├── test_row_softmax.py     # Stage 5 测试脚本（独立验收依据）
├── MEMORY.md               # 多位智能体共享的记忆：思路、决策、调试过程（跨智能体协作）
└── .orchestrator_state.json # 状态机文件：当前阶段、重试次数
```

#### 第二步：逐个认识关键产物


下表说明每份产物「是什么（用大白话说）、由谁产出、该重点检查什么」。你会发现它们正好对应上节介绍的一个个智能体——每种产物都来自一个阶段、由一位明确的责任人产出，检查它也就等于检查那位责任人干得怎么样：

> **常见疑问**：`SPEC.md` 和 `DESIGN.md` 有什么区别？`SPEC.md` 回答「**要做什么**」——需求层面的规格，如算子算什么、输入输出、精度标准；`DESIGN.md` 回答「**怎么做**」——设计层面的方案，如数据怎么分块、循环怎么写、内存怎么安排。前者是任务定义的延续，后者才是工程师风格的规划。

#### 第三步：独立验收

验收不能只看 Agent 在对话里的一句话总结——模型可能把「应该通过」说成「已经通过」，也可能不小心漏掉某个失败的细节。必须由学习者自己动手，独立执行测试：

```bash
cd pypto-gym/cannbot-skills/plugins-official/pypto-op-orchestrator  # 进入部署好的 Agent 工程目录
python custom/row_softmax/test_row_softmax.py
```

这三条通过标准，**缺一不可**：

1. 测试**真实运行在 Ascend NPU 上**（不是 CPU、仿真，也不是「实现和自身比较」偷换概念）；
2. 每个输出均 `all_close=true`（`atol=1e-3`、`rtol=1e-3`），即实现结果与 golden 逐值对比都落在容差内；
3. 进程**退出码为 0**（命令正常结束，没有报错中断）。

> **为什么不直接信 Agent 的报告？** 报告的结论可能源于模型「预期」而非真实运行，也可能来自中间某个未完成的步骤。独立重跑等于把结论重新验一遍——这也是整套工程机制的核心理念：**不看总结，只信证据**。

> **说明**：本章使用 `atol=1e-3`、`rtol=1e-3` 作为容差标准，这是考虑到 Agent 生成代码可能存在合理数值波动。对于手工开发的高精度算子，通常可以使用更严格的容差（如 `1e-5` 或 `1e-6`）。

</div>

<div align="left">

## 课后练习

本节练习用于复盘 PyPTO Agent 的职责分工、Lint Gate 门禁与共享状态机制。单选题只有一个正确答案，多选题有两个或以上正确答案，填空题请填写正确的术语或文件名。

1. （单选题）Stage 5 中某个模块的 NPU 校验失败了。按照 PyPTO Agent 的职责分工，正确的处理方式是哪一项？  
   A. Coder 直接修改代码，然后自己宣布通过  
   B. Verifier 判定失败并归类故障 → Debugger 定位根因并给出补丁方案 → Coder 应用补丁 → Verifier 重新校验，通过后才进入下一模块  
   C. Orchestrator 亲自接手调试并修复 Kernel  
   D. 先记录失败，跳过这个模块做下一个
2. （多选题）Lint Gate（自动门禁）从哪些维度检查代码质量？（选择所有适用项）  
   A. D1 框架合规性：装饰器、张量签名 Shape、JIT 相关规范  
   B. D2 交付物完整性：各阶段要求产出的文档与代码是否齐备  
   C. D3 文件隔离：实现文件禁止引入 Torch，golden 文件禁止调用 PyPTO  
   D. D4 测试规范：测试用例覆盖、误差容差配置  
   E. D5 跨文件一致性：模块接口约定与实现代码是否匹配  
   F. D6 性能基准：代码执行时间是否达标
3. （单选题）为什么 Lint Gate 必须在**写入阶段**强制阻断，而不是事后检查？  
   A. 事后检查会增加额外的计算开销  
   B. 写入阶段阻断可以防止违规代码落盘污染后续流程，事后巡检无法阻止已写入的非法产物  
   C. 写入阶段阻断比事后检查更容易实现  
   D. 事后检查无法读取代码内容
4. （填空题）PyPTO Agent 每个算子工作目录下有两份持续更新的共享状态文件：____________ 是供人阅读的叙事记录，记录设计思路、决策依据、检查凭证与调试过程；____________ 由机器自动维护，记录当前阶段、重试次数、模块状态与产物哈希，是从中断点恢复的依据。
5. （填空题）Stage 5 某模块 NPU 校验失败后，按职责分工应依次由____________判定失败并归类故障，由____________定位根因并给出补丁方案，再由____________应用补丁，最后仍由____________重新校验，通过后才进入下一模块。

**执行以下代码获取答案。**

</div>

In [ ]:
!cat ./answer/05.02_answer.txt

<div align="left">

## 本节小结：一张知识框架图

本节把 PyPTO Agent 的「团队结构、工作流程、支撑体系、产出物、使用要点」串成一张可复习的图。

### 一、团队结构：一个总控 + 八个智能体

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">角色</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">主要阶段</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">一句话职责</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Orchestrator（总控）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">全程</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">流程编排，不写代码、不判结果</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Planner（规划）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 1</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">需求 → `SPEC.md`、`API_REPORT.md`</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Mathematician（数学）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 2</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编写 golden 参考实现</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Architect（架构）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 3</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">复杂度评估与分块设计</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Designer（设计）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 4</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">模块拆分与接口约定</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Coder（编码）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 5</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编写 PyPTO kernel 实现</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Verifier（校验）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 5/6</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">独立判定 PASS/FAIL 并归类故障</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Debugger（调试）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 5</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">定位根因、给出补丁方案</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Optimizer（优化）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Stage 7</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">性能采集与调优</td></tr></tbody></table>

### 二、工作流程：Stage 1–7 状态机

Stage 1 需求规划 → Stage 2 算法基准 → Stage 3 架构设计 → Stage 4 模块接口设计 → Stage 5 按模块编码闭环 → Stage 6 最终 E2E 验证 → Stage 7 性能调优。每阶段由对应智能体完成并产出指定文件，通过门禁后才进入下一阶段；Stage 7 仅在任务明确要求性能目标时进入。

### 三、支撑体系：技能、状态与门禁

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">组件</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">作用</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">通俗比喻</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Skills 技能库</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">按需加载的阶段知识</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">工具箱：用哪个拿哪个</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">`MEMORY.md` + `.orchestrator_state.json`</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">记录进度与依据、支持断点续跑</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">工作笔记 + 进度表</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Lint Gate 自动门禁</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">写入关键文件时强制拦截违规</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">门卫：进门先检查</td></tr></tbody></table>

### 四、核心教训与使用要点

1. **职责分离**：编写实现（Coder）与结果判定（Verifier）严格分开，实现者不得自行宣布通过；
2. **失败链路**：Verifier 判定失败并归类故障 → Debugger 定位根因 → Coder 应用补丁 → Verifier 重新校验；
3. **证据优先**：不看 Agent 的总结，只信独立重跑出来的结果；
4. **任务完整**：用五要素（名称公式、输入输出、支持范围、参考测试、停止条件）把任务定义清楚；
5. **可断点续跑**：进度写入状态文件，中断后可在交互式界面里继续。

下一节用行 Softmax 把这一整套流程完整实践一遍。

</div>